In [ ]:
!nvidia-smi
import sys, torch
print("Python :", sys.version)
print("PyTorch :", torch.__version__, "| CUDA dispo :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

Thu Nov 19 16:02:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             34W /   70W |     147MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os

REPO_DIR = "/content/scanDDM"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/phuselab/scanDDM.git {REPO_DIR}
else:
    print("Le dépôt est déjà cloné :", REPO_DIR)

%cd {REPO_DIR}

Le dépôt est déjà cloné : /content/scanDDM
/content/scanDDM


In [ ]:
# Option A : Installation rapide dans l'environnement Colab actuel
# On ignore torch/torchvision/torchaudio s'ils sont déjà installés avec CUDA sur Colab
import pathlib, subprocess, sys

req_path = pathlib.Path("requirements.txt")
lines = [l.strip() for l in req_path.read_text().splitlines() if l.strip() and not l.strip().startswith("#")]

filtered = []
for line in lines:
    pkg = line.split("==")[0].split(">=")[0].split("<=")[0].strip().lower()
    if pkg in {"torch", "torchvision", "torchaudio"}:
        print(f"Ignoré (version CUDA Colab conservée) : {line}")
        continue
    # On retire les pins stricts si besoin pour éviter les conflits avec le Python de Colab
    filtered.append(pkg)

print("Paquets à installer :", filtered)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + filtered)
print("Installation Option A terminée.")

In [ ]:
# Option B (uniquement si l'Option A échoue) : Environnement isolé Python 3.9 + PyTorch 2.3 via uv
USE_PY39_VENV = False  # Passe à True uniquement en cas de conflit de dépendances

if USE_PY39_VENV:
    !pip install -q uv
    !uv venv /content/venv_scanddm --python 3.9
    !uv pip install --python /content/venv_scanddm/bin/python torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu121
    !uv pip install --python /content/venv_scanddm/bin/python -r requirements.txt
    PYTHON_BIN = "/content/venv_scanddm/bin/python"
else:
    import sys
    PYTHON_BIN = sys.executable

print("Interpréteur sélectionné :", PYTHON_BIN)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls "/content/drive/MyDrive"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
'Bio test'	    data-zip		  Perso  'Worldskills - 2025'
'Colab Notebooks'  'Dossier sans titre'   Umons


In [ ]:
!unzip -q "/content/drive/MyDrive/data-zip/data.zip" -d /content/scanDDM/data

In [18]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# === 0. PATCH AUTO NUMPY + IMPORT DE LA CLASSE scanDDM ===
%cd /content/scanDDM
if "/content/scanDDM" not in sys.path:
    sys.path.insert(0, "/content/scanDDM")

scan_py = Path("/content/scanDDM/scanDDM.py")
src = scan_py.read_text(encoding="utf-8")
old_line = "curr_fix = (int(x),int(y))"
new_line = "curr_fix = (int(x.item()) if hasattr(x, 'item') else int(x), int(y.item()) if hasattr(y, 'item') else int(y))"
if old_line in src:
    scan_py.write_text(src.replace(old_line, new_line), encoding="utf-8")
    print("Patch NumPy appliqué sur scanDDM.py")

for mod in list(sys.modules):
    if mod == "scanDDM" or mod.startswith("scanDDM."):
        del sys.modules[mod]

from scanDDM import scanDDM

# === 1. PARAMÈTRES ===
INPUT_DIR = Path("/content/scanDDM/data/data/scenes/layoutA_TA")
os.makedirs("/content/results", exist_ok=True)
OUTPUT_DIR = Path("/content/results")  # ou "/content/drive/MyDrive/ISIA106/scanddm_vis"
PROMPT = "keyboard"
N_OBS = 10
SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# === 2. INITIALISATION DU MODÈLE (1 SEULE FOIS) ===
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device utilisé :", device)

model = scanDDM(
    experiment_dur=4.5,   # Plus de temps pour atteindre les bords (2.0 par défaut)
    fps=50,               # Pas de temps plus fin (25 par défaut)
    threshold=1.5,        # Lisse le bruit (1.0 par défaut)
    noise=3.0,            # Moins aléatoire (7 par défaut)
    kappa=30,             # Pénalise moins les longues saccades (10 par défaut)
    eta=25,               # Renforce la prior_map (17 par défaut)
    device=device,
)

# === 3. LISTER LES IMAGES ===
exts = {".jpg", ".jpeg", ".png"}
image_paths = sorted(p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in exts)
print(f"{len(image_paths)} images trouvées.")

plt.ioff()

# === 4. BOUCLE DE GÉNÉRATION DES IMAGES ===
for img_path in tqdm(image_paths[:60], desc=f"Visu ScanDDM ({PROMPT})"):
    out_img = OUTPUT_DIR / f"{img_path.stem}_{PROMPT}.png"
    if out_img.exists():
        continue

    set_seed(SEED)
    img = np.array(Image.open(img_path).convert("RGB"))

    scans, prior_map = model.simulate_scanpaths(
        image=img,
        prompt=[PROMPT],
        n_observers=N_OBS,
    )

    if hasattr(prior_map, "detach"):
        prior_np = prior_map.detach().cpu().numpy()
    else:
        prior_np = np.asarray(prior_map)
    prior_np = np.squeeze(prior_np)

    # --- Création de la figure (2 panneaux) ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(img)
    axes[0].imshow(prior_np, cmap="jet", alpha=0.45)
    axes[0].set_title(f"Prior map ({PROMPT})")
    axes[0].axis("off")

    axes[1].imshow(img)
    for sp in scans:
        arr = np.atleast_2d(np.asarray(sp))
        if arr.size == 0:
            continue
        axes[1].scatter(arr[:, 0], arr[:, 1], c="yellow", s=25, alpha=0.35, edgecolors="none")

    if len(scans) > 0:
        first = np.atleast_2d(np.asarray(scans[0]))
        if first.size > 0:
            xs, ys = first[:, 0], first[:, 1]
            if first.shape[1] >= 3:
                d = first[:, 2].astype(float)
                sizes = 60 + 240 * (d / (d.max() + 1e-8))
            else:
                sizes = 90
            axes[1].plot(xs, ys, "-o", color="red", linewidth=1.8, markersize=4, alpha=0.85)
            axes[1].scatter(xs, ys, s=sizes, facecolors="none", edgecolors="cyan", linewidths=2)
            for idx, (x, y) in enumerate(zip(xs, ys), 1):
                axes[1].text(x, y, str(idx), color="white", fontsize=8, fontweight="bold",
                             ha="center", va="center")

    axes[1].set_title(f"Fixations ({N_OBS} obs) + 1 scanpath")
    axes[1].axis("off")

    plt.tight_layout()
    fig.savefig(out_img, dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Terminé ! Images sauvegardées dans : {OUTPUT_DIR}")

/content/scanDDM
Device utilisé : cuda
205 images trouvées.


Visu ScanDDM (keyboard):   0%|          | 0/60 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/462 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/462 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/462 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/462 [00:00<?, ?it/s]

... (53 outputs left out)

Loading weights:   0%|          | 0/462 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/462 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/462 [00:00<?, ?it/s]

Terminé ! Images sauvegardées dans : /content/results


In [17]:
!rm -f /content/results/*.png

In [ ]:
import glob
from IPython.display import Image, display

imgs = sorted(glob.glob("content/scanddm_vis/*.png"))
if not imgs:
    print("Aucune figure capturée dans demo_outputs/. Vérifie la sortie de la cellule précédente.")
for img_path in imgs:
    print(img_path)
    display(Image(filename=img_path))